In [1]:
# --- COMPLETE CELL 1 ---
import pickle
import numpy as np
import os
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

print("Step 1: Loading data...")
with open("../data/clean/cc_data.pkl", "rb") as f:
    X_cc_train, X_cc_test, y_cc_train, y_cc_test = pickle.load(f)

with open("../data/clean/hc_data.pkl", "rb") as f:
    X_hc_train, X_hc_test, y_hc_train, y_hc_test, _ = pickle.load(f)
print("✅ Data loaded.")

print("\nStep 2: Loading Agent 2 (Random Forest)...")
if os.path.exists("../models/agent2_rf.pkl"):
    with open("../models/agent2_rf.pkl", "rb") as f:
        rf_model = pickle.load(f)
    print("✅ RF Model loaded.")
else:
    print("❌ ERROR: agent2_rf.pkl not found! You MUST go to 04_agent2_rf.ipynb, click 'Run All', and wait for it to finish before running this cell.")

print("\nStep 3: Loading Agent 1 (RNN)...")
if os.path.exists("../models/agent1_rnn.h5"):
    rnn_model = tf.keras.models.load_model("../models/agent1_rnn.h5", compile=False)
    print("✅ RNN Model loaded.")
else:
    print("❌ ERROR: agent1_rnn.h5 not found! Run 03_agent1_rnn.ipynb first.")

print("\n🎉 All steps completed. You can now move to Cell 2.")

Step 1: Loading data...
✅ Data loaded.

Step 2: Loading Agent 2 (Random Forest)...
✅ RF Model loaded.

Step 3: Loading Agent 1 (RNN)...
✅ RNN Model loaded.

🎉 All steps completed. You can now move to Cell 2.


In [2]:
# Generate Base Predictions (Probabilities)
# Reshape CC data for RNN
X_cc_test_rnn = X_cc_test.values.reshape((X_cc_test.shape[0], 1, X_cc_test.shape[1]))

# Get probabilities from Agent 1 (RNN)
rnn_probs = rnn_model.predict(X_cc_test_rnn).flatten()

# Get probabilities from Agent 2 (RF)
rf_probs = rf_model.predict_proba(X_hc_test)[:, 1]

print("Agent 1 (RNN) predictions shape:", rnn_probs.shape)
print("Agent 2 (RF) predictions shape:", rf_probs.shape)

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step
Agent 1 (RNN) predictions shape: (56962,)
Agent 2 (RF) predictions shape: (61503,)


In [3]:
# Create Meta-Features for Agent 3
# We truncate to the minimum length so the arrays match up
min_len = min(len(rnn_probs), len(rf_probs))

meta_X = np.column_stack((rnn_probs[:min_len], rf_probs[:min_len]))
meta_y = y_hc_test[:min_len].values

print("Meta-feature shape:", meta_X.shape)
print("Meta-target shape:", meta_y.shape)

Meta-feature shape: (56962, 2)
Meta-target shape: (56962,)


In [4]:
# Train Agent 3
from sklearn.model_selection import train_test_split
X_meta_train, X_meta_test, y_meta_train, y_meta_test = train_test_split(
    meta_X, meta_y, test_size=0.3, random_state=42, stratify=meta_y
)

agent3_model = LogisticRegression(class_weight='balanced')
agent3_model.fit(X_meta_train, y_meta_train)

print("Agent 3 (Logistic Regression) trained successfully!")

Agent 3 (Logistic Regression) trained successfully!


In [5]:
# Evaluate
y_pred_a3 = agent3_model.predict(X_meta_test)
y_proba_a3 = agent3_model.predict_proba(X_meta_test)[:, 1]

print("--- Agent 3 (Final Decision) Classification Report ---")
print(classification_report(y_meta_test, y_pred_a3))
print("Final AUC-ROC:", roc_auc_score(y_meta_test, y_proba_a3))

# Save
import os
os.makedirs("../models", exist_ok=True)
with open("../models/agent3_lr.pkl", "wb") as f:
    pickle.dump(agent3_model, f)
print("\n✅ Saved Agent 3 model to ../models/agent3_lr.pkl")

--- Agent 3 (Final Decision) Classification Report ---
              precision    recall  f1-score   support

           0       0.96      0.68      0.80     15710
           1       0.15      0.65      0.25      1379

    accuracy                           0.68     17089
   macro avg       0.56      0.67      0.52     17089
weighted avg       0.89      0.68      0.75     17089

Final AUC-ROC: 0.7306829873768065

✅ Saved Agent 3 model to ../models/agent3_lr.pkl
